# MobileNet — Etapa 1: Tratamento de Dados

Este notebook cobre todas as etapas de pré-processamento do dataset de classificação de faixa de preço de celulares, desde a análise exploratória até a preparação dos dados para o treinamento do MLP.

---

**Variável alvo:** `price_range`

| Classe | Descrição        |
|--------|------------------|
| 0      | Baixo custo      |
| 1      | Custo médio      |
| 2      | Alto custo       |
| 3      | Custo muito alto |

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

IMAGES_DIR = 'images'
os.makedirs(IMAGES_DIR, exist_ok=True)

---
## 2. Carregamento dos Dados

In [ ]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

print(f'Train: {train.shape[0]} amostras, {train.shape[1]} colunas')
print(f'Test:  {test.shape[0]} amostras, {test.shape[1]} colunas')
train.head()

---
## 3. Análise Exploratória (EDA)

### 3.1 Tipos e valores ausentes

O primeiro passo é verificar os tipos de cada coluna e a presença de valores ausentes, que podem comprometer o treinamento do modelo.

In [ ]:
train.info()

In [ ]:
ausentes = train.isnull().sum()
print('Valores ausentes por coluna:')
print(ausentes[ausentes > 0] if ausentes.any() else 'Nenhum valor ausente encontrado.')

### 3.2 Estatísticas descritivas

As estatísticas descritivas fornecem um resumo das principais medidas para cada feature:

- **Média** $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$
- **Desvio padrão** $s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2}$
- **Mínimo, Quartis e Máximo** — indicam a dispersão e possíveis outliers

In [ ]:
train.describe().round(2)

### 3.3 Distribuição da variável alvo

É importante verificar se as classes estão balanceadas. Um dataset desbalanceado pode enviesar o modelo a prever as classes mais frequentes.

In [ ]:
contagem = train['price_range'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(contagem.index, contagem.values, color=sns.color_palette('steelblue', 4))
ax.bar_label(bars, padding=4, fontsize=11)
ax.set_title('Distribuição da Variável Alvo (price_range)', fontsize=13)
ax.set_xlabel('Classe')
ax.set_ylabel('Quantidade')
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['0 – Baixo', '1 – Médio', '2 – Alto', '3 – Muito alto'])
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/distribuicao_target.png', dpi=150)
plt.show()
print('\nProporção por classe:')
print((contagem / len(train) * 100).round(2).astype(str) + '%')

### 3.4 Correlação entre features

O coeficiente de correlação de Pearson mede a relação linear entre duas variáveis:

$$r_{xy} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2 \cdot \sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

- $r = 1$: correlação positiva perfeita
- $r = -1$: correlação negativa perfeita
- $r = 0$: sem correlação linear

Features com alta correlação entre si podem introduzir redundância no modelo.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 11))
sns.heatmap(
    train.corr(),
    annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, ax=ax, annot_kws={'size': 8}
)
ax.set_title('Mapa de Correlação de Pearson', fontsize=14)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/correlacao.png', dpi=150)
plt.show()

In [ ]:
# Features com maior correlação com a variável alvo
corr_target = train.corr()['price_range'].drop('price_range').sort_values(key=abs, ascending=False)
print('Correlação com price_range (ordenada):')
print(corr_target.round(3).to_string())

### 3.5 Distribuição das features por classe

Boxplots permitem visualizar como as features se distribuem em cada classe, ajudando a identificar quais variáveis têm maior poder discriminativo.

In [ ]:
features_destaque = corr_target.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(features_destaque):
    sns.boxplot(data=train, x='price_range', y=feat, ax=axes[i],
                palette='coolwarm')
    axes[i].set_title(f'{feat}', fontsize=11)
    axes[i].set_xlabel('Classe')

fig.suptitle('Top 6 Features — Distribuição por Classe', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/boxplots_features.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Separação de Features e Alvo

In [ ]:
X = train.drop(columns='price_range')
y = train['price_range']

print(f'Features (X): {X.shape}')
print(f'Classes  (y): {sorted(y.unique())}')

---
## 5. Normalização — StandardScaler

O MLP é sensível à escala das features. O **StandardScaler** transforma cada feature para ter média zero e desvio padrão unitário:

$$z = \frac{x - \mu}{\sigma}$$

Onde:
- $x$ é o valor original
- $\mu$ é a média da feature no conjunto de treino
- $\sigma$ é o desvio padrão da feature no conjunto de treino
- $z$ é o valor normalizado

> **Importante:** o scaler é ajustado (`fit`) **somente nos dados de treino** e aplicado (`transform`) tanto no treino quanto na validação, evitando vazamento de informação (*data leakage*).

In [ ]:
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)

print(f'Treino:    {X_train.shape}')
print(f'Validação: {X_val.shape}')

In [ ]:
# Verificação: média ≈ 0 e desvio padrão ≈ 1 no treino
df_check = pd.DataFrame(X_train, columns=X.columns)
print('Média (primeiras 5 features):')
print(df_check.mean().head().round(6).to_string())
print('\nDesvio padrão (primeiras 5 features):')
print(df_check.std().head().round(6).to_string())

---
## 6. Divisão Treino / Validação

Utilizamos a divisão **80% treino / 20% validação** com estratificação para garantir que a proporção de cada classe seja preservada nos dois conjuntos.

A estratificação é especialmente importante quando as classes são balanceadas — como é o caso deste dataset — evitando que alguma classe fique sub-representada na validação.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (nome, serie) in zip(axes, [('Treino', y_train), ('Validação', y_val)]):
    counts = serie.value_counts().sort_index()
    bars = ax.bar(counts.index, counts.values, color=sns.color_palette('steelblue', 4))
    ax.bar_label(bars, padding=3, fontsize=10)
    ax.set_title(f'Distribuição — {nome}', fontsize=12)
    ax.set_xlabel('Classe')
    ax.set_ylabel('Quantidade')
    ax.set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/distribuicao_split.png', dpi=150)
plt.show()

---
## 7. Resumo do Pré-processamento

| Etapa                  | Detalhe                                        |
|------------------------|------------------------------------------------|
| Valores ausentes       | Nenhum                                         |
| Tipos de dados         | Todos numéricos (int64 / float64)              |
| Normalização           | StandardScaler (µ=0, σ=1)                     |
| Divisão                | 80% treino / 20% validação (stratify=y)        |
| Amostras de treino     | 1600                                           |
| Amostras de validação  | 400                                            |
| Número de features     | 20                                             |
| Número de classes      | 4                                              |

Os dados estão prontos para o treinamento do MLP na próxima etapa.